In [3]:
import numpy as np
from PIL import Image


def safe_phash(image, hash_size=16, highfreq_factor=4):
    """
    Pure numpy pHash (perceptual hash) implementation.
    Does NOT use imagehash or scipy. No ufunc issues.
    """

    img = image.convert("L").resize(
        (hash_size * highfreq_factor, hash_size * highfreq_factor),
        Image.Resampling.LANCZOS
    )

    pixels = np.asarray(img, dtype=np.float32)

    # Perform DCT (2D)
    dct = np.fft.fft2(pixels)   # using FFT instead of DCT2
    dct_lowfreq = np.abs(dct[:hash_size, :hash_size])

    med = np.median(dct_lowfreq)
    diff = dct_lowfreq > med

    # Convert to hex hash
    return "".join(
        f"{int(''.join(str(int(b)) for b in diff[i:i+4]), 2):01x}"
        for i in range(0, hash_size * hash_size, 4)
    )

In [5]:
# Hamming distance between 2 hex hashes
def hamming_distance_hex(h1, h2):
    b1 = bin(int(h1, 16))[2:].zfill(256)
    b2 = bin(int(h2, 16))[2:].zfill(256)
    return sum(c1 != c2 for c1, c2 in zip(b1, b2))

In [10]:
import os
import json
from PIL import Image
import imagehash

# --------------------------
# CONFIG: Set your folders
# --------------------------
FOLDER_A = "mathpix_dsf_images"  # Mathpix images
FOLDER_B = "pymupdf_dsf_images"  # PyMuPDF images

# --------------------------
# Helper: Load and compute perceptual hash
# --------------------------
def compute_hashes(folder):
    hashes = {}
    for file in sorted(os.listdir(folder)):
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            path = os.path.join(folder, file)
            try:
                img = Image.open(path)
                h = imagehash.phash(img)
                hashes[file] = h
            except Exception as e:
                print(f"[ERROR] Could not process {path}: {e}")
    return hashes


# --------------------------
# Step 1: Compute hashes
# --------------------------
print("Computing perceptual hashes...")
hashes_A = compute_hashes(FOLDER_A)
hashes_B = compute_hashes(FOLDER_B)
print(f"Images in A: {len(hashes_A)}, Images in B: {len(hashes_B)}")

# --------------------------
# Step 2: Compare hashes
# --------------------------
matches = []

for a_file, a_hash in hashes_A.items():
    best_match = None
    best_dist = float("inf")

    for b_file, b_hash in hashes_B.items():
        dist = a_hash - b_hash  # Hamming distance
        if dist < best_dist:
            best_dist = dist
            best_match = b_file

    matches.append({
        "A_image": a_file,
        "B_best_match": best_match,
        "distance": int(best_dist)
    })

# --------------------------
# Step 3: Print results
# --------------------------
print("\n=== MATCH RESULTS (sorted by distance) ===")
matches_sorted = sorted(matches, key=lambda x: x["distance"])

for m in matches_sorted:
    print(f"{m['A_image']}  <--->  {m['B_best_match']}   (dist={m['distance']})")

# --------------------------
# Step 4: Save to JSON
# --------------------------
output_path = "dsf_matches_m1.json"
with open(output_path, "w") as f:
    json.dump(matches_sorted, f, indent=4)

print(f"\nSaved detailed matches to {output_path}")


Computing perceptual hashes...
Images in A: 172, Images in B: 178

=== MATCH RESULTS (sorted by distance) ===
2025_03_17_ca60ec0bfd96dcf8e028g-165.jpg  <--->  page164_img00.jpeg   (dist=0)
2025_03_17_ca60ec0bfd96dcf8e028g-212.jpg  <--->  page211_img00.jpeg   (dist=0)
2025_03_17_ca60ec0bfd96dcf8e028g-024.jpg  <--->  page023_img00.png   (dist=2)
2025_03_17_ca60ec0bfd96dcf8e028g-028.jpg  <--->  page027_img00.jpeg   (dist=2)
2025_03_17_ca60ec0bfd96dcf8e028g-144(1).jpg  <--->  page143_img00.jpeg   (dist=2)
2025_03_17_ca60ec0bfd96dcf8e028g-023.jpg  <--->  page022_img00.jpeg   (dist=4)
2025_03_17_ca60ec0bfd96dcf8e028g-031.jpg  <--->  page030_img00.jpeg   (dist=4)
2025_03_17_ca60ec0bfd96dcf8e028g-063(2).jpg  <--->  page062_img01.jpeg   (dist=4)
2025_03_17_ca60ec0bfd96dcf8e028g-102.jpg  <--->  page101_img00.jpeg   (dist=4)
2025_03_17_ca60ec0bfd96dcf8e028g-162.jpg  <--->  page152_img00.png   (dist=4)
2025_03_17_ca60ec0bfd96dcf8e028g-164.jpg  <--->  page163_img00.png   (dist=4)
2025_03_17_ca60ec0

In [ ]:
import json
import os
from PIL import Image
import matplotlib.pyplot as plt

# --------------------------
# CONFIG: update your directories
# --------------------------
FOLDER_A = "mathpix_dsf_images"  # Mathpix images
FOLDER_B = "pymupdf_dsf_images"  # PyMuPDF images
JSON_FILE = "dsf_matches_m1.json"

# --------------------------
# Load the JSON
# --------------------------
with open(JSON_FILE, "r") as f:
    pairs = json.load(f)

print(f"Loaded {len(pairs)} image pairs.")

# --------------------------
# Visualize each pair
# --------------------------
for pair in pairs:
    a_name = pair["A_image"]
    b_name = pair["B_best_match"]
    distance = pair["distance"]

    a_path = os.path.join(FOLDER_A, a_name)
    b_path = os.path.join(FOLDER_B, b_name)

    if not os.path.exists(a_path) or not os.path.exists(b_path):
        print(f"Skipping missing file: {a_path} or {b_path}")
        continue

    imgA = Image.open(a_path)
    imgB = Image.open(b_path)

    # Create a figure
    plt.figure(figsize=(12, 5))
    plt.suptitle(f"{a_name}  <-->  {b_name}  (distance={distance})", fontsize=14)

    # Left = A_image
    plt.subplot(1, 2, 1)
    plt.imshow(imgA)
    plt.title("Image A")
    plt.axis("off")

    # Right = B_image
    plt.subplot(1, 2, 2)
    plt.imshow(imgB)
    plt.title("Best Match (Image B)")
    plt.axis("off")

    plt.show()


In [13]:
"""
compare_and_find_missing.py

Purpose:
 - Compare two folders of images (A and B) using perceptual hashing (pHash).
 - Find missing/extra images, many->one and one->many mapping patterns.
 - Output JSON summary and an HTML report with thumbnails for manual inspection.

Dependencies:
 pip install pillow imagehash numpy tqdm

Optional (for OCR-based fallback checks):
 pip install pytesseract opencv-python

Usage:
 python compare_and_find_missing.py
"""

import os
import json
from PIL import Image
import imagehash
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from html import escape

# ----------------------
# CONFIG
# ----------------------
FOLDER_A = "mathpix_pdf_dsf_images"  # Mathpix images
FOLDER_B = "pymupdf_dsf_images"  # PyMuPDF images
OUTPUT_JSON = "compare_summary_mp.json"
REPORT_HTML = "compare_report_mp.html"

# perceptual-hash threshold: <= good match. tweak if needed.
PHASH_THRESHOLD = 8

# Max number of candidate B images to keep per A for ambiguity check
TOP_K = 5

# Thumbnail size for HTML report
THUMBNAIL_SIZE = (320, 400)

# ----------------------
# Helpers
# ----------------------
def list_images(folder):
    files = sorted([f for f in os.listdir(folder) if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".tiff"))])
    return files

def compute_phashes(folder, files):
    phashes = {}
    sizes = {}
    for fname in tqdm(files, desc=f"Hashing {folder}"):
        path = os.path.join(folder, fname)
        try:
            img = Image.open(path).convert("RGB")
            ph = imagehash.phash(img)   # perceptual hash
            phashes[fname] = ph
            sizes[fname] = img.size
        except Exception as e:
            print(f"[ERROR] Cannot open {path}: {e}")
    return phashes, sizes

def hamming_dist(h1, h2):
    return int(h1 - h2)

# ----------------------
# Load & hash
# ----------------------
os.makedirs(FOLDER_A, exist_ok=True)
os.makedirs(FOLDER_B, exist_ok=True)

files_A = list_images(FOLDER_A)
files_B = list_images(FOLDER_B)

print(f"Found {len(files_A)} images in A and {len(files_B)} images in B.")

ph_A, sizes_A = compute_phashes(FOLDER_A, files_A)
ph_B, sizes_B = compute_phashes(FOLDER_B, files_B)

# Sanity check
if not ph_A or not ph_B:
    print("No images found or hashing failed. Check folders and file types.")
    exit(1)

# ----------------------
# Compute distance matrix (A x B)
# ----------------------
print("Computing pairwise distances...")
dist_matrix = {}
for a_name, a_hash in tqdm(ph_A.items(), desc="A→B distances"):
    row = []
    for b_name, b_hash in ph_B.items():
        row.append((b_name, hamming_dist(a_hash, b_hash)))
    # sort by distance ascending
    row_sorted = sorted(row, key=lambda x: x[1])
    dist_matrix[a_name] = row_sorted

# Also compute best matches from B→A
dist_matrix_B = {}
for b_name, b_hash in tqdm(ph_B.items(), desc="B→A distances"):
    row = []
    for a_name, a_hash in ph_A.items():
        row.append((a_name, hamming_dist(a_hash, b_hash)))
    row_sorted = sorted(row, key=lambda x: x[1])
    dist_matrix_B[b_name] = row_sorted

# ----------------------
# Determine matches with threshold
# ----------------------
matches_A_to_B = {}
for a_name, row in dist_matrix.items():
    best_b, best_dist = row[0]
    matches_A_to_B[a_name] = {"best_B": best_b, "dist": best_dist, "candidates": row[:TOP_K]}

matches_B_to_A = {}
for b_name, row in dist_matrix_B.items():
    best_a, best_dist = row[0]
    matches_B_to_A[b_name] = {"best_A": best_a, "dist": best_dist, "candidates": row[:TOP_K]}

# classify matched vs unmatched
unmatched_in_B = []  # A images with no good B match -> likely missing in B
for a, info in matches_A_to_B.items():
    if info["dist"] > PHASH_THRESHOLD:
        unmatched_in_B.append({"A": a, "best_B": info["best_B"], "dist": info["dist"], "candidates": info["candidates"]})

unmatched_in_A = []  # B images with no good A match -> likely extra in B
for b, info in matches_B_to_A.items():
    if info["dist"] > PHASH_THRESHOLD:
        unmatched_in_A.append({"B": b, "best_A": info["best_A"], "dist": info["dist"], "candidates": info["candidates"]})

# many->one and one->many
b_to_as = defaultdict(list)
a_to_bs = defaultdict(list)
for a, info in matches_A_to_B.items():
    if info["dist"] <= PHASH_THRESHOLD:
        b = info["best_B"]
        b_to_as[b].append((a, info["dist"]))
        a_to_bs[a].append((b, info["dist"]))

many_to_one = {b: lst for b, lst in b_to_as.items() if len(lst) > 1}  # multiple As matching same B
one_to_many = {a: lst for a, lst in a_to_bs.items() if len(lst) > 1}  # multiple Bs matching same A (rare)

# ambiguous: best and second best close
ambiguous = []
for a, info in matches_A_to_B.items():
    cand = info["candidates"]
    if len(cand) >= 2:
        if (cand[1][1] - cand[0][1]) <= 2:  # threshold for closeness: tweakable
            ambiguous.append({"A": a, "candidates": cand[:3]})

# summary
summary = {
    "counts": {"A": len(files_A), "B": len(files_B)},
    "phash_threshold": PHASH_THRESHOLD,
    "unmatched_in_B (A with no good match in B)": unmatched_in_B,
    "unmatched_in_A (B with no good match in A)": unmatched_in_A,
    "many_to_one_B_to_As": many_to_one,
    "one_to_many_A_to_Bs": one_to_many,
    "ambiguous_candidates": ambiguous,
}

# Save JSON
with open(OUTPUT_JSON, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Wrote summary JSON to {OUTPUT_JSON}")

# ----------------------
# Build a simple HTML report for quick visual inspection
# ----------------------
def make_thumbnail(src_path, dest_path, size=THUMBNAIL_SIZE):
    try:
        img = Image.open(src_path).convert("RGB")
        img.thumbnail(size)
        img.save(dest_path, "JPEG", quality=75)
        return True
    except Exception as e:
        print(f"[WARN] thumbnail failed for {src_path}: {e}")
        return False

report_entries = []

# include unmatched A (missing in B)
for item in summary["unmatched_in_B (A with no good match in B)"][:200]:
    a = item["A"]
    b = item["best_B"]
    d = item["dist"]
    pair_id = f"missA__{a}"
    thumbA = os.path.join("report_thumbs", f"{pair_id}_A.jpg")
    thumbB = os.path.join("report_thumbs", f"{pair_id}_B.jpg")
    os.makedirs("report_thumbs", exist_ok=True)
    make_thumbnail(os.path.join(FOLDER_A, a), thumbA)
    if os.path.exists(os.path.join(FOLDER_B, b)):
        make_thumbnail(os.path.join(FOLDER_B, b), thumbB)
    report_entries.append({
        "type": "missing_in_B",
        "A": a, "B": b, "dist": d, "thumbA": thumbA, "thumbB": thumbB
    })

# include unmatched B (extra in B)
for item in summary["unmatched_in_A (B with no good match in A)"][:200]:
    b = item["B"]
    a = item["best_A"]
    d = item["dist"]
    pair_id = f"extraB__{b}"
    thumbA = os.path.join("report_thumbs", f"{pair_id}_A.jpg")
    thumbB = os.path.join("report_thumbs", f"{pair_id}_B.jpg")
    make_thumbnail(os.path.join(FOLDER_B, b), thumbB)
    if os.path.exists(os.path.join(FOLDER_A, a)):
        make_thumbnail(os.path.join(FOLDER_A, a), thumbA)
    report_entries.append({
        "type": "extra_in_B",
        "A": a, "B": b, "dist": d, "thumbA": thumbA, "thumbB": thumbB
    })

# include many->one
for b, lst in list(many_to_one.items())[:200]:
    pair_id = f"manytoone__{b}"
    thumbs = []
    make_thumbnail(os.path.join(FOLDER_B, b), os.path.join("report_thumbs", f"{pair_id}_B.jpg"))
    for idx, (a, dist) in enumerate(lst):
        t = os.path.join("report_thumbs", f"{pair_id}_A{idx}.jpg")
        make_thumbnail(os.path.join(FOLDER_A, a), t)
        thumbs.append({"A": a, "dist": dist, "thumb": t})
    report_entries.append({
        "type": "many_to_one",
        "B": b,
        "matches": thumbs,
        "thumbB": os.path.join("report_thumbs", f"{pair_id}_B.jpg")
    })

# ambiguous entries
for amb in ambiguous[:200]:
    a = amb["A"]
    pair_id = f"amb__{a}"
    make_thumbnail(os.path.join(FOLDER_A, a), os.path.join("report_thumbs", f"{pair_id}_A.jpg"))
    cands = []
    for idx, (b, dist) in enumerate(amb["candidates"]):
        t = os.path.join("report_thumbs", f"{pair_id}_B{idx}.jpg")
        make_thumbnail(os.path.join(FOLDER_B, b), t)
        cands.append({"B": b, "dist": dist, "thumb": t})
    report_entries.append({"type": "ambiguous", "A": a, "candidates": cands, "thumbA": os.path.join("report_thumbs", f"{pair_id}_A.jpg")})

# create HTML
html_lines = [
    "<!doctype html>",
    "<html><head><meta charset='utf-8'><title>Image compare report</title></head><body>",
    "<h1>Image compare report</h1>",
    f"<p>Counts: A={len(files_A)}, B={len(files_B)}. phash threshold={PHASH_THRESHOLD}</p>",
    "<hr/>"
]

for entry in report_entries:
    t = entry["type"]
    if t == "missing_in_B" or t == "extra_in_B":
        html_lines.append(f"<div style='margin:20px;padding:10px;border:1px solid #ccc;'>")
        html_lines.append(f"<h3>{escape(t)} — dist={entry['dist']}</h3>")
        html_lines.append("<table><tr>")
        # A
        html_lines.append(f"<td><b>A:</b><div>{escape(entry['A'])}</div>")
        if os.path.exists(entry['thumbA']):
            html_lines.append(f"<img src='{entry['thumbA']}'/></td>")
        else:
            html_lines.append(f"<div style='color:#999'>thumb missing</div></td>")
        # B
        html_lines.append(f"<td><b>B:</b><div>{escape(entry['B'])}</div>")
        if os.path.exists(entry['thumbB']):
            html_lines.append(f"<img src='{entry['thumbB']}'/></td>")
        else:
            html_lines.append(f"<div style='color:#999'>thumb missing</div></td>")
        html_lines.append("</tr></table></div>")
    elif t == "many_to_one":
        html_lines.append(f"<div style='margin:20px;padding:10px;border:2px solid #c33;'>")
        html_lines.append(f"<h3>MANY→ONE: B = {escape(entry['B'])}</h3>")
        html_lines.append("<table><tr>")
        if os.path.exists(entry["thumbB"]):
            html_lines.append(f"<td><img src='{entry['thumbB']}'/><div><b>B:</b>{escape(entry['B'])}</div></td>")
        html_lines.append("<td>")
        for m in entry["matches"]:
            html_lines.append("<div style='display:inline-block;margin:6px;text-align:center;'>")
            if os.path.exists(m["thumb"]):
                html_lines.append(f"<img src='{m['thumb']}'/><div>{escape(m['A'])}<br/>(dist={m['dist']})</div>")
            else:
                html_lines.append(f"<div style='color:#999'>{escape(m['A'])} (thumb missing)</div>")
            html_lines.append("</div>")
        html_lines.append("</td></tr></table></div>")
    elif t == "ambiguous":
        html_lines.append(f"<div style='margin:20px;padding:10px;border:1px dashed #666;'>")
        html_lines.append(f"<h3>Ambiguous candidates for A = {escape(entry['A'])}</h3>")
        if os.path.exists(entry["thumbA"]):
            html_lines.append(f"<div><img src='{entry['thumbA']}'/></div>")
        for cand in entry["candidates"]:
            html_lines.append(f"<div style='display:inline-block;margin:6px;text-align:center;'>")
            if os.path.exists(cand["thumb"]):
                html_lines.append(f"<img src='{cand['thumb']}'/><div>{escape(cand['B'])}<br/>(dist={cand['dist']})</div>")
            else:
                html_lines.append(f"<div style='color:#999'>{escape(cand['B'])}</div>")
            html_lines.append("</div>")
        html_lines.append("</div>")

html_lines.append("<hr><p>Generated by compare_and_find_missing.py</p>")
html_lines.append("</body></html>")

with open(REPORT_HTML, "w", encoding="utf-8") as f:
    f.write("\n".join(html_lines))

print(f"HTML report written to {REPORT_HTML}. Thumbnails in ./report_thumbs/")
print("Done.")


Found 172 images in A and 178 images in B.


Hashing pymupdf_dsf_images: 100%|██████████| 178/178 [00:00<00:00, 420.94it/s]


Computing pairwise distances...


B→A distances: 100%|██████████| 178/178 [00:00<00:00, 4824.37it/s]


Wrote summary JSON to compare_summary_mp.json
HTML report written to compare_report_mp.html. Thumbnails in ./report_thumbs/
Done.


In [8]:
"""
compare_ssim.py

Compare images in two folders using SSIM (Structural Similarity Index).
Produces:
 - matches.json : list of A_image -> top_k B candidates with SSIM scores
 - optionally visual comparison images saved to out_vis/
Dependencies:
 - pillow
 - opencv-python
 - scikit-image
 - numpy
 - tqdm
Install: pip install pillow opencv-python scikit-image numpy tqdm
"""

import os
import json
from PIL import Image, ImageOps
import numpy as np
import cv2
from skimage.metrics import structural_similarity as ssim
from tqdm import tqdm

# -----------------------
# CONFIG
# -----------------------
FOLDER_A = "mathpix_dsf_images"  # Mathpix images
FOLDER_B = "pymupdf_dsf_images"  # PyMuPDF images
OUTPUT_JSON = "dsf_matches.json"
OUT_VIS_DIR = "out_vis"    # set to None to skip saving visual comparisons
RESIZE_TO = (800, 800)     # (width, height) target size for SSIM comparison
TOP_K = 3                  # how many top candidates from B to keep per A
MIN_SSIM = 0.3             # optional: filter out matches below this SSIM
VERBOSE = True

# -----------------------
# HELPERS
# -----------------------
def load_image_as_gray(path, target_size=RESIZE_TO):
    """
    Load an image, convert to grayscale, resize while preserving aspect ratio,
    and pad to target_size (width, height).
    Returns a uint8 numpy array of shape (H, W).
    """
    img = Image.open(path).convert("RGB")
    # preserve aspect ratio and fit into target box
    img.thumbnail(target_size, Image.LANCZOS)
    # create new image and paste centered
    background = Image.new("RGB", target_size, (255, 255, 255))
    offset = ((target_size[0] - img.width) // 2, (target_size[1] - img.height) // 2)
    background.paste(img, offset)
    gray = background.convert("L")
    arr = np.array(gray, dtype=np.uint8)
    return arr

def compute_ssim(img_arr1, img_arr2):
    """
    Compute SSIM between two grayscale uint8 arrays.
    """
    # scikit-image's ssim expects either float images or uint8; pass data_range for uint8
    score = ssim(img_arr1, img_arr2, data_range=img_arr2.max() - img_arr2.min())
    return float(score)

def make_vis(a_path, b_path, score, out_path, target_size=RESIZE_TO):
    """
    Save a side-by-side visual comparison with SSIM score as text.
    """
    a = Image.open(a_path).convert("RGB")
    b = Image.open(b_path).convert("RGB")

    # Resize/pad for nicer visualization (same routine)
    def fit(img):
        img.thumbnail(target_size, Image.LANCZOS)
        bg = Image.new("RGB", target_size, (255,255,255))
        off = ((target_size[0]-img.width)//2, (target_size[1]-img.height)//2)
        bg.paste(img, off)
        return bg

    A = fit(a)
    B = fit(b)
    combined = Image.new("RGB", (target_size[0]*2 + 10, target_size[1] + 40), (255,255,255))
    combined.paste(A, (0, 0))
    combined.paste(B, (target_size[0] + 10, 0))

    # draw text: file names and score
    import PIL.ImageDraw as ImageDraw
    import PIL.ImageFont as ImageFont
    draw = ImageDraw.Draw(combined)
    font = None
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 18)
    except Exception:
        font = ImageFont.load_default()

    # Text positions
    draw.text((5, target_size[1] + 5), f"A: {os.path.basename(a_path)}", fill=(0,0,0), font=font)
    draw.text((target_size[0] + 15, target_size[1] + 5), f"B: {os.path.basename(b_path)}", fill=(0,0,0), font=font)
    draw.text((5, target_size[1] + 25), f"SSIM: {score:.4f}", fill=(0,0,0), font=font)

    combined.save(out_path)

# -----------------------
# MAIN
# -----------------------
def main():
    # validate folders
    if not os.path.isdir(FOLDER_A):
        raise FileNotFoundError(f"Folder A not found: {FOLDER_A}")
    if not os.path.isdir(FOLDER_B):
        raise FileNotFoundError(f"Folder B not found: {FOLDER_B}")

    a_files = sorted([f for f in os.listdir(FOLDER_A) if f.lower().endswith((".png",".jpg",".jpeg",".tiff",".bmp"))])
    b_files = sorted([f for f in os.listdir(FOLDER_B) if f.lower().endswith((".png",".jpg",".jpeg",".tiff",".bmp"))])

    if len(a_files) == 0 or len(b_files) == 0:
        raise RuntimeError("One of the folders is empty or contains no supported image files.")

    # Preload B images to avoid repeated disk reads
    if VERBOSE: print(f"Loading and preprocessing {len(b_files)} images from B...")
    b_cache = {}
    for bf in tqdm(b_files, desc="Preloading B", disable=not VERBOSE):
        path = os.path.join(FOLDER_B, bf)
        try:
            b_cache[bf] = load_image_as_gray(path)
        except Exception as e:
            print(f"[WARN] Could not load B image {bf}: {e}")

    results = []
    if OUT_VIS_DIR:
        os.makedirs(OUT_VIS_DIR, exist_ok=True)

    if VERBOSE: print(f"Comparing {len(a_files)} A-images against {len(b_cache)} B-images...")
    for af in tqdm(a_files, desc="Comparing", disable=not VERBOSE):
        a_path = os.path.join(FOLDER_A, af)
        try:
            a_arr = load_image_as_gray(a_path)
        except Exception as e:
            print(f"[WARN] Could not load A image {af}: {e}")
            continue

        # compute SSIM against every B
        scores = []
        for bf, b_arr in b_cache.items():
            try:
                score = compute_ssim(a_arr, b_arr)
            except Exception:
                # fallback: convert to float and provide explicit data_range
                score = ssim(a_arr.astype(np.float32), b_arr.astype(np.float32), data_range=255.0)
            scores.append((bf, score))

        # sort by SSIM desc
        scores_sorted = sorted(scores, key=lambda x: x[1], reverse=True)
        topk = scores_sorted[:TOP_K]
        # optional filtering by MIN_SSIM
        topk_filtered = [ {"B_image": bfn, "ssim": float(s)} for (bfn,s) in topk if s >= MIN_SSIM ]

        # Save result
        entry = {
            "A_image": af,
            "top_matches": topk_filtered if len(topk_filtered)>0 else [{"B_image": topk[0][0], "ssim": float(topk[0][1])}]  # always keep 1 best
        }
        results.append(entry)

        # Optionally write visualizations
        if OUT_VIS_DIR:
            for i,(bfn, s) in enumerate(topk):
                vis_name = f"{os.path.splitext(af)[0]}__vs__{os.path.splitext(bfn)[0]}__rank{i+1}__ssim{int(s*10000)}.png"
                vis_path = os.path.join(OUT_VIS_DIR, vis_name)
                try:
                    make_vis(a_path, os.path.join(FOLDER_B, bfn), s, vis_path)
                except Exception as e:
                    # non-fatal
                    if VERBOSE:
                        print(f"[WARN] Could not save visualization for {af} vs {bfn}: {e}")

    # Save JSON
    with open(OUTPUT_JSON, "w") as f:
        json.dump(results, f, indent=2)

    print(f"\nDone. Results saved to {OUTPUT_JSON}")
    if OUT_VIS_DIR:
        print(f"Visual comparisons saved to {OUT_VIS_DIR} (one file per top-K candidate).")

if __name__ == "__main__":
    main()


Loading and preprocessing 178 images from B...


Preloading B: 100%|██████████| 178/178 [00:00<00:00, 299.40it/s]


Comparing 172 A-images against 178 B-images...


Comparing: 100%|██████████| 172/172 [12:56<00:00,  4.52s/it]


Done. Results saved to dsf_matches.json
Visual comparisons saved to out_vis (one file per top-K candidate).


In [ ]:
# -----------------------
# CONFIG
# -----------------------
FOLDER_A = "mathpix_dsf_images"  # Mathpix images
FOLDER_B = "pymupdf_dsf_images"  # PyMuPDF images
OUTPUT_JSON = "dsf_matches.json"
OUT_VIS_DIR = "out_vis"    # set to None to skip saving visual comparisons
RESIZE_TO = (800, 800)     # (width, height) target size for SSIM comparison
TOP_K = 3                  # how many top candidates from B to keep per A
MIN_SSIM = 0.3             # optional: filter out matches below this SSIM
VERBOSE = True


main()